In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


# =========================
# Настройки
# =========================

DIAGRAMS_PATH = Path("persistence_diagrams_31_columns.csv")

OUTPUT_DIR = Path("persistence_vectors")
OUTPUT_DIR.mkdir(exist_ok=True)

# Какие размерности гомологий учитывать.
# Если считались только H0 и H1, оставляем [0, 1].
# Если добавишь H2, поставь [0, 1, 2].
HOMOLOGY_DIMS = [0, 1]

# Сетка для landscapes, Betti curves, Euler curves
N_GRID = 100

# Persistence Landscape: сколько слоев брать
# landscape layer 1 = самая большая "палатка" в каждой точке сетки,
# layer 2 = вторая по величине и т.д.
N_LANDSCAPE_LAYERS = 5

# Persistence Image
PI_BIRTH_BINS = 20
PI_PERSISTENCE_BINS = 20
PI_SIGMA = 0.05


# =========================
# Загрузка диаграмм
# =========================

diagrams = pd.read_csv(DIAGRAMS_PATH)

required_columns = {
    "column",
    "tau",
    "dimension",
    "homology_dimension",
    "birth",
    "death",
    "persistence",
    "is_infinite",
}

missing = required_columns - set(diagrams.columns)
if missing:
    raise ValueError(f"В файле не хватает колонок: {missing}")

# Убираем бесконечные интервалы, потому что большинство векторизаций
# работают только с конечными birth-death точками.
finite = diagrams[
    np.isfinite(diagrams["birth"])
    & np.isfinite(diagrams["death"])
    & np.isfinite(diagrams["persistence"])
    & (diagrams["persistence"] > 0)
].copy()

finite["homology_dimension"] = finite["homology_dimension"].astype(int)

print("Всего точек диаграмм:", len(diagrams))
print("Конечных точек диаграмм:", len(finite))
print("Рядов:", finite["column"].nunique())
print("Размерности гомологий:", sorted(finite["homology_dimension"].unique()))

## Общие функции для векторизации

In [ ]:
def get_common_grid(df, n_grid=100):
    """
    Общая сетка по birth/death для всех диаграмм.
    На этой сетке строятся landscapes, Betti curves и Euler curves.
    """
    min_birth = df["birth"].min()
    max_death = df["death"].max()

    if not np.isfinite(min_birth) or not np.isfinite(max_death):
        raise ValueError("Некорректные birth/death значения")

    if min_birth == max_death:
        max_death = min_birth + 1.0

    return np.linspace(min_birth, max_death, n_grid)


def get_diagram_points(df, column_name, homology_dim):
    """
    Возвращает точки диаграммы для одного ряда и одной размерности гомологии.
    Формат: массив shape = (n_points, 2), где колонки birth, death.
    """
    part = df[
        (df["column"] == column_name)
        & (df["homology_dimension"] == homology_dim)
    ]

    if part.empty:
        return np.empty((0, 2))

    return part[["birth", "death"]].to_numpy(dtype=float)


def safe_stats(values):
    """
    Набор устойчивых статистик для массива.
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return {
            "count": 0,
            "sum": 0.0,
            "mean": 0.0,
            "std": 0.0,
            "min": 0.0,
            "max": 0.0,
            "median": 0.0,
        }

    return {
        "count": len(values),
        "sum": float(np.sum(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "min": float(np.min(values)),
        "max": float(np.max(values)),
        "median": float(np.median(values)),
    }

## Persistence Landscapes

Идея: каждая точка диаграммы (birth, death) превращается в треугольную функцию. Потом в каждой точке сетки берутся самые большие значения этих функций.

In [ ]:
def persistence_landscape(points, grid, n_layers=5):
    """
    Persistence Landscape для одной диаграммы.

    points: массив birth-death точек shape = (n_points, 2)
    grid: сетка значений epsilon
    n_layers: сколько слоев landscape брать

    Возвращает вектор длины n_layers * len(grid).
    """
    if len(points) == 0:
        return np.zeros(n_layers * len(grid))

    birth = points[:, 0]
    death = points[:, 1]

    tents = []

    for b, d in zip(birth, death):
        if not np.isfinite(b) or not np.isfinite(d) or d <= b:
            continue

        # Треугольная функция:
        # lambda(t) = max(0, min(t - birth, death - t))
        values = np.maximum(0.0, np.minimum(grid - b, d - grid))
        tents.append(values)

    if len(tents) == 0:
        return np.zeros(n_layers * len(grid))

    tents = np.vstack(tents)

    # Сортируем значения в каждой точке сетки по убыванию
    sorted_values = np.sort(tents, axis=0)[::-1]

    layers = []

    for layer_idx in range(n_layers):
        if layer_idx < sorted_values.shape[0]:
            layers.append(sorted_values[layer_idx])
        else:
            layers.append(np.zeros(len(grid)))

    return np.concatenate(layers)

## Persistence Images

Идея: точка (birth, death) переводится в (birth, persistence), затем каждая точка размазывается гауссианом по сетке. В итоге получается картинка, которую можно развернуть в вектор.

In [ ]:
def persistence_image(
    points,
    birth_range,
    persistence_range,
    birth_bins=20,
    persistence_bins=20,
    sigma=0.05,
):
    """
    Persistence Image для одной диаграммы.

    Возвращает вектор длины birth_bins * persistence_bins.
    """
    image = np.zeros((persistence_bins, birth_bins), dtype=float)

    if len(points) == 0:
        return image.ravel()

    b_min, b_max = birth_range
    p_min, p_max = persistence_range

    if b_min == b_max:
        b_max = b_min + 1.0

    if p_min == p_max:
        p_max = p_min + 1.0

    birth_grid = np.linspace(b_min, b_max, birth_bins)
    persistence_grid = np.linspace(p_min, p_max, persistence_bins)

    birth_mesh, persistence_mesh = np.meshgrid(birth_grid, persistence_grid)

    for b, d in points:
        if not np.isfinite(b) or not np.isfinite(d) or d <= b:
            continue

        p = d - b

        # Вес можно выбрать разный.
        # Здесь вес = persistence, чтобы более устойчивые признаки имели больший вклад.
        weight = p

        gaussian = np.exp(
            -(
                (birth_mesh - b) ** 2
                + (persistence_mesh - p) ** 2
            )
            / (2 * sigma**2)
        )

        image += weight * gaussian

    return image.ravel()

## Betti Curves

Кривая Бетти показывает, сколько топологических объектов живо при каждом значении масштаба.

In [ ]:
def betti_curve(points, grid):
    """
    Betti curve для одной диаграммы.

    В каждой точке grid считаем количество интервалов [birth, death),
    которые сейчас живы.
    """
    curve = np.zeros(len(grid), dtype=float)

    if len(points) == 0:
        return curve

    for b, d in points:
        if not np.isfinite(b) or not np.isfinite(d) or d <= b:
            continue

        alive = (grid >= b) & (grid < d)
        curve[alive] += 1.0

    return curve

## Euler Curves

Кривая Эйлера считается через кривые Бетти:

χ(t) = β0(t) - β1(t) + β2(t) - β3(t) + ...

In [ ]:
def euler_curve(betti_curves_by_dim, homology_dims):
    """
    Euler curve по набору Betti curves.

    betti_curves_by_dim: dict, где ключ — размерность гомологии,
    значение — Betti curve.
    """
    result = None

    for h_dim in homology_dims:
        curve = betti_curves_by_dim.get(h_dim)

        if curve is None:
            continue

        sign = (-1) ** h_dim

        if result is None:
            result = sign * curve.copy()
        else:
            result += sign * curve

    if result is None:
        first_dim = homology_dims[0]
        return np.zeros_like(betti_curves_by_dim[first_dim])

    return result

## Характеристики диаграмм ?

Считаем простые признаки для каждой диаграммы: число точек, сумма persistence, максимум persistence, среднее, стандартное отклонение и т.д.

In [ ]:
def diagram_features(points):
    """
    Числовые характеристики одной persistence diagram.
    """
    if len(points) == 0:
        return {
            "point_count": 0,
            "total_persistence": 0.0,
            "mean_persistence": 0.0,
            "std_persistence": 0.0,
            "max_persistence": 0.0,
            "median_persistence": 0.0,
            "mean_birth": 0.0,
            "mean_death": 0.0,
            "max_death": 0.0,
        }

    birth = points[:, 0]
    death = points[:, 1]
    persistence = death - birth

    persistence = persistence[np.isfinite(persistence) & (persistence > 0)]

    if len(persistence) == 0:
        return {
            "point_count": 0,
            "total_persistence": 0.0,
            "mean_persistence": 0.0,
            "std_persistence": 0.0,
            "max_persistence": 0.0,
            "median_persistence": 0.0,
            "mean_birth": 0.0,
            "mean_death": 0.0,
            "max_death": 0.0,
        }

    return {
        "point_count": int(len(persistence)),
        "total_persistence": float(np.sum(persistence)),
        "mean_persistence": float(np.mean(persistence)),
        "std_persistence": float(np.std(persistence)),
        "max_persistence": float(np.max(persistence)),
        "median_persistence": float(np.median(persistence)),
        "mean_birth": float(np.mean(birth)),
        "mean_death": float(np.mean(death)),
        "max_death": float(np.max(death)),
    }

## Сбор векторизации в таблицы

In [ ]:
# Общая сетка по birth/death для всех диаграмм
grid = get_common_grid(finite, n_grid=N_GRID)

# Диапазоны для persistence image
birth_range = (
    float(finite["birth"].min()),
    float(finite["birth"].max()),
)

persistence_range = (
    float(finite["persistence"].min()),
    float(finite["persistence"].max()),
)

columns = sorted(finite["column"].dropna().unique())

landscape_rows = []
image_rows = []
betti_rows = []
euler_rows = []
features_rows = []
combined_rows = []

for column_name in columns:
    print("Processing:", column_name)

    landscape_feature_dict = {"column": column_name}
    image_feature_dict = {"column": column_name}
    betti_feature_dict = {"column": column_name}
    features_dict = {"column": column_name}

    betti_curves_for_euler = {}

    combined_dict = {"column": column_name}

    for h_dim in HOMOLOGY_DIMS:
        points = get_diagram_points(finite, column_name, h_dim)

        # -------------------------
        # Persistence Landscape
        # -------------------------
        landscape_vec = persistence_landscape(
            points,
            grid,
            n_layers=N_LANDSCAPE_LAYERS,
        )

        for i, value in enumerate(landscape_vec):
            key = f"landscape_H{h_dim}_{i}"
            landscape_feature_dict[key] = value
            combined_dict[key] = value

        # -------------------------
        # Persistence Image
        # -------------------------
        image_vec = persistence_image(
            points,
            birth_range=birth_range,
            persistence_range=persistence_range,
            birth_bins=PI_BIRTH_BINS,
            persistence_bins=PI_PERSISTENCE_BINS,
            sigma=PI_SIGMA,
        )

        for i, value in enumerate(image_vec):
            key = f"pimage_H{h_dim}_{i}"
            image_feature_dict[key] = value
            combined_dict[key] = value

        # -------------------------
        # Betti Curve
        # -------------------------
        betti_vec = betti_curve(points, grid)
        betti_curves_for_euler[h_dim] = betti_vec

        for i, value in enumerate(betti_vec):
            key = f"betti_H{h_dim}_{i}"
            betti_feature_dict[key] = value
            combined_dict[key] = value

        # -------------------------
        # Diagram Features
        # -------------------------
        features = diagram_features(points)

        for feature_name, value in features.items():
            key = f"H{h_dim}_{feature_name}"
            features_dict[key] = value
            combined_dict[key] = value

    # -------------------------
    # Euler Curve
    # -------------------------
    euler_vec = euler_curve(betti_curves_for_euler, HOMOLOGY_DIMS)

    euler_feature_dict = {"column": column_name}

    for i, value in enumerate(euler_vec):
        key = f"euler_{i}"
        euler_feature_dict[key] = value
        combined_dict[key] = value

    landscape_rows.append(landscape_feature_dict)
    image_rows.append(image_feature_dict)
    betti_rows.append(betti_feature_dict)
    euler_rows.append(euler_feature_dict)
    features_rows.append(features_dict)
    combined_rows.append(combined_dict)


landscape_df = pd.DataFrame(landscape_rows)
image_df = pd.DataFrame(image_rows)
betti_df = pd.DataFrame(betti_rows)
euler_df = pd.DataFrame(euler_rows)
features_df = pd.DataFrame(features_rows)
combined_df = pd.DataFrame(combined_rows)

print("landscape_df:", landscape_df.shape)
print("image_df:", image_df.shape)
print("betti_df:", betti_df.shape)
print("euler_df:", euler_df.shape)
print("features_df:", features_df.shape)
print("combined_df:", combined_df.shape)

## Сохранение результатов

In [ ]:
landscape_df.to_csv(OUTPUT_DIR / "persistence_landscapes.csv", index=False)
image_df.to_csv(OUTPUT_DIR / "persistence_images.csv", index=False)
betti_df.to_csv(OUTPUT_DIR / "betti_curves.csv", index=False)
euler_df.to_csv(OUTPUT_DIR / "euler_curves.csv", index=False)
features_df.to_csv(OUTPUT_DIR / "persistence_diagram_features.csv", index=False)
combined_df.to_csv(OUTPUT_DIR / "persistence_all_vectorizations.csv", index=False)

print("Saved to:", OUTPUT_DIR)